# Lab 0 — L'ambiente di lavoro

## Prima di iniziare

**Teoria di riferimento**

- Blocco 0.1, «The publish/subscribe arrangement»: il nodo pubblica, un broker inoltra a chi si è sottoscritto.
- Blocco 0.1, «Topic names and wildcards»: i nomi dei topic e i filtri con `+` e `#`.
- Blocco 0.1, «QoS 0, 1 and 2 exchanges»: quali pacchetti si scambiano client e broker.

**Che cosa vediamo oggi**

1. Un broker MQTT sul portatile, un sottoscrittore e un pubblicatore da terminale.
2. Lo stesso scambio da Python, con la libreria `paho-mqtt`, ricevuto sia dal terminale sia dal notebook.
3. Il repository del corso con Git: dove sono le cose, che cosa è cambiato, come si annulla una modifica.

**Tempo**: 45 minuti. Terminali e verifica 15, Parte 1 5, Parte 2 10, Parte 3 10, chiusura 5.

Il dataset del corso non serve oggi: si scarica e si guarda nel Lab 2.

### Dove va questo laboratorio

Il portatile è pronto per il corso, e un messaggio passa attraverso il broker?

| | domanda | dove arriva |
|---|---|---|
| Parte 1 | Le librerie e il broker ci sono? | Una tabella: ogni riga dice una versione, o che il broker risponde. |
| Parte 2 | Un messaggio mandato da Python arriva a chi lo aspetta? | Sì, se il filtro copre il topic; il QoS ricevuto è il minimo dei due. |
| Parte 3 | Dove sono le cose nel repository, e come si annulla una modifica? | labs/ è del docente, work/ è nostra; git restore riporta un file alla versione del docente. |

### Tre terminali

Si aprono tre terminali, tutti nella radice del repository e con l'ambiente virtuale
attivo (`source .venv/bin/activate`; su Windows `.venv\Scripts\activate`).

**Terminale 1 — il broker.**

```
mosquitto -v
```

Avvia Mosquitto in primo piano sulla porta 1883; `-v` stampa ogni pacchetto che il broker
riceve e invia. Senza file di configurazione, Mosquitto 2.x accetta connessioni solo da
`localhost`: per il corso basta. Se il comando risponde `Address already in use`, un broker
è già in esecuzione come servizio: va bene, si usa quello e il terminale 1 resta libero.

**Terminale 2 — un sottoscrittore.**

```
mosquitto_sub -h localhost -t 'test/#' -v
```

Si sottoscrive a tutti i topic che iniziano con `test/`; `-v` stampa il topic prima del
messaggio. Il comando non termina: resta in attesa.

**Terminale 3 — un pubblicatore.**

```
mosquitto_pub -h localhost -t test/ciao -m "ciao"
```

Nel terminale 2 compare `test/ciao ciao`; nel terminale 1 il broker registra CONNECT,
PUBLISH e DISCONNECT del pubblicatore.

**Windows.** L'installatore di mosquitto.org mette i programmi in
`C:\Program Files\mosquitto\` e registra il servizio «Mosquitto Broker»; se il servizio
è avviato, la porta 1883 è già occupata e il broker c'è. La cartella non è nel `PATH`: o si aggiunge, o si lancia
`& "C:\Program Files\mosquitto\mosquitto_sub.exe" ...` da PowerShell. Per vedere il log
con `mosquitto -v` si ferma prima il servizio (`net stop mosquitto`, da amministratore). In
`cmd.exe` il filtro va fra virgolette doppie: `-t "test/#"`.

**macOS.** `brew install mosquitto`. Il broker sta in `$(brew --prefix)/sbin`
(`/opt/homebrew/sbin` su Apple Silicon, `/usr/local/sbin` su Intel), che può non essere nel
`PATH`; `brew services start mosquitto` lo avvia come servizio, con la configurazione in
`$(brew --prefix)/etc/mosquitto/mosquitto.conf`.

**Linux.** `sudo apt install mosquitto mosquitto-clients`: il servizio parte da solo
(`systemctl status mosquitto`), quindi `mosquitto -v` trova la porta occupata. O si usa il
servizio, o lo si ferma con `sudo systemctl stop mosquitto`.

### La verifica

Nel terminale 3, dalla radice del repository:

```
python scripts/verify_setup.py --quick
```

Controlla i pacchetti installati, quale copia di `iomtlab` viene importata e il broker, poi
fa girare i test (un minuto circa). Scrive il rapporto in `verifica.txt`, nella radice:
se un controllo fallisce, è il file da mandare al docente.

### Setup del notebook

In [ ]:
import importlib
import subprocess
import sys
import threading
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from iomtlab.config import REPO_ROOT
from iomtlab.net import broker_available, mqtt_client

WORK = REPO_ROOT / "work"
WORK.mkdir(exist_ok=True)
print("repository:", REPO_ROOT)

## Parte 1 — Versioni e servizi

> **Domanda.** Le librerie e il broker ci sono? **Dove arriva.** Una tabella: ogni riga dice una versione, o che il broker risponde.

Il notebook usa le librerie installate nell'ambiente `.venv`, la copia di `iomtlab` di questo
repository e il broker sulla porta 1883. Controllo le tre cose in una tabella: se una riga dice `ASSENTE`,
il problema è nell'ambiente, non nel codice dei laboratori. Teoria: Blocco 0.1,
«The publish/subscribe arrangement» (il broker a cui il nodo pubblica) e «Scope, transport and
versions» (la porta 1883 e il trasporto TCP).

In [ ]:
PACCHETTI = ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "torch", "paho.mqtt", "iomtlab"]

def controlla(host, port):
    righe = [("python", sys.version.split()[0])]
    for nome in PACCHETTI:
        try:
            m = importlib.import_module(nome)
            righe.append((nome, getattr(m, "__version__", "?")))
        except ImportError:
            righe.append((nome, "ASSENTE"))
    import iomtlab
    da = Path(iomtlab.__file__).resolve().parent
    radice = REPO_ROOT.resolve()
    if da.is_relative_to(radice):
        righe.append(("iomtlab da", f"./{da.relative_to(radice).as_posix()}"))
    else:
        righe.append(("iomtlab da", f"{da} (fuori dal repository)"))
    # il broker: provo solo ad aprire una connessione TCP sulla porta
    righe.append((f"broker {host}:{port}", "risponde" if broker_available(host, port) else "ASSENTE"))
    return pd.DataFrame(righe, columns=["che cosa", "stato"]).set_index("che cosa")

controlla("localhost", 1883)

### Prova a cambiare

Si cambiano solo i valori fra le due righe `PARAMETRI`, poi si riesegue la cella.

**Prima di rieseguire: che cosa ti aspetti?** Sulla porta 1884 non ascolta nessuno: che cosa scrive la riga del broker, e perché la connessione fallisce subito invece di attendere?

In [ ]:
# ── PARAMETRI ──────────────────────────────────────────────────────────────
BROKER_HOST = "localhost"   # dove cerco il broker: "localhost" o "127.0.0.1"
BROKER_PORT = 1884          # la porta: 1883 è quella di MQTT; 8883 è MQTT su TLS; prova 1884
# ───────────────────────────────────────────────────────────────────────────
controlla(BROKER_HOST, BROKER_PORT)

**Che cosa guardare.** Solo la riga del broker cambia. La connessione a una porta chiusa di `localhost` viene rifiutata immediatamente dal sistema operativo (un segmento RST), quindi la cella non aspetta il timeout di un secondo.

**Da discutere**

1. Se il broker gira su un'altra macchina della rete, quale dei due parametri cambi, e che cosa deve permettere la configurazione di Mosquitto 2.x?
2. La riga del broker dice solo che la porta accetta una connessione TCP: basta per dire che dall'altra parte c'è un broker MQTT? Che cosa servirebbe per esserne certi?

## Parte 2 — Un messaggio da Python

> **Domanda.** Un messaggio mandato da Python arriva a chi lo aspetta? **Dove arriva.** Sì, se il filtro copre il topic; il QoS ricevuto è il minimo dei due.

Stesso scambio dei terminali 2 e 3, ma da Python. Un client del notebook si sottoscrive con
un filtro e raccoglie i messaggi; un secondo client pubblica. Se il terminale 2 è ancora
aperto, il messaggio compare anche lì: il broker consegna a tutti i sottoscrittori il cui
filtro corrisponde al topic. Teoria: Blocco 0.1, «Topic names and wildcards».

In [ ]:
def un_messaggio(topic, testo, qos_pub=0, qos_sub=0, filtro="test/#", attesa_s=2.0):
    arrivati = []
    arrivato = threading.Event()

    def on_message(client, userdata, msg):
        # il broker mi consegna il messaggio: mi segno topic, testo, QoS e l'istante di arrivo
        arrivati.append({"topic": msg.topic, "testo": msg.payload.decode(),
                         "qos": msg.qos, "t": time.perf_counter()})
        arrivato.set()

    sub = mqtt_client("notebook-sub", 1883, on_message=on_message)
    sub.subscribe(filtro, qos=qos_sub)
    time.sleep(0.3)
    pub = mqtt_client("notebook-pub", 1883)
    t0 = time.perf_counter()
    pub.publish(topic, testo, qos=qos_pub).wait_for_publish()
    arrivato.wait(attesa_s)
    for c in (pub, sub):
        c.disconnect(); c.loop_stop()
    if not arrivati:
        print(f"nessun messaggio su '{filtro}' entro {attesa_s} s")
        return None
    m = arrivati[0]
    m["ritardo_ms"] = round(1000 * (m.pop("t") - t0), 2)
    return m

un_messaggio("test/python", "ciao da Python")

### Prova a cambiare

Si cambiano solo i valori fra le due righe `PARAMETRI`, poi si riesegue la cella.

**Prima di rieseguire: che cosa ti aspetti?** Con topic `prova/python` e filtro `test/#`, il messaggio arriva al notebook? E al terminale 2? Con `FILTRO = "#"`, con quale QoS arriva un messaggio pubblicato a QoS 2 su una sottoscrizione a QoS 0?

In [ ]:
# ── PARAMETRI ──────────────────────────────────────────────────────────────
TOPIC = "prova/python"      # il topic su cui pubblico; prova "test/python" e "prova/python"
MESSAGGIO = "ciao di nuovo" # il testo, qualunque stringa
QOS_PUB = 2                 # QoS del pubblicatore: 0, 1 o 2
QOS_SUB = 0                 # QoS del sottoscrittore: 0, 1 o 2
FILTRO = "test/#"           # il filtro del notebook; prova "test/+", "#", "prova/#"
# ───────────────────────────────────────────────────────────────────────────
un_messaggio(TOPIC, MESSAGGIO, QOS_PUB, QOS_SUB, FILTRO)

**Che cosa guardare.** Un topic che il filtro non copre non arriva a nessuno e il broker non lo segnala: il pubblicatore non sa chi riceve. Il campo `qos` del messaggio ricevuto è il minimo fra QoS di pubblicazione e QoS di sottoscrizione. Il ritardo su `localhost` è dell'ordine del millisecondo: su una rete reale lo decide il collegamento.

**Da discutere**

1. Il pubblicatore non riceve errori quando nessuno è sottoscritto al topic. In un sistema di monitoraggio, come si accorge chi riceve che un nodo ha smesso di pubblicare?
2. Perché il broker consegna a QoS 0 un messaggio pubblicato a QoS 2, se la sottoscrizione è a QoS 0?

## Parte 3 — Il repository con Git

> **Domanda.** Dove sono le cose nel repository, e come si annulla una modifica? **Dove arriva.** labs/ è del docente, work/ è nostra; git restore riporta un file alla versione del docente.

Il repository è la cartella del corso, con tutta la sua storia: ogni versione salvata dal
docente è un *commit*. Git tiene traccia di che cosa cambia rispetto all'ultimo commit. Le
celle qui sotto lanciano gli stessi comandi che si danno nel terminale: `git log` mostra gli
ultimi commit, `git status` i file cambiati.

In [ ]:
def git(*argomenti):
    # lancio un comando git nella cartella del repository e stampo quello che risponde
    r = subprocess.run(["git", *argomenti], cwd=REPO_ROOT, capture_output=True, text=True)
    print(f"$ git {' '.join(argomenti)}")
    print(r.stdout.strip() or r.stderr.strip() or "(nessun output: niente da segnalare)")
    print()

git("log", "--oneline", "-5")
git("status", "--short")

Le cartelle principali: `labs/` contiene i notebook del docente e non si modifica; `work/` è
la propria cartella di lavoro, ignorata da Git; `src/iomtlab/` è la libreria del corso;
`scripts/` contiene la verifica dell'ambiente e, dal Lab 2, lo script che scarica il dataset.

In [ ]:
for cartella in ["labs", "work", "src/iomtlab", "scripts", "projects"]:
    n = len(list((REPO_ROOT / cartella).glob("*")))
    print(f"{cartella:14} {n:3} elementi")

### Prova a cambiare

Si cambiano solo i valori fra le due righe `PARAMETRI`, poi si riesegue la cella.

**Prima di rieseguire: che cosa ti aspetti?** Che cosa scrive `git status` dopo aver modificato un file di `labs/`? E dopo aver creato un file in `work/`? Che cosa fa `git restore` sul file di `labs/`?

In [ ]:
# ── PARAMETRI ──────────────────────────────────────────────────────────────
FILE_LABS = "labs/README.md"     # un file del docente: lo modifico e poi lo ripristino
FILE_WORK = "work/prova.txt"     # un file mio, in work/
# ───────────────────────────────────────────────────────────────────────────
with open(REPO_ROOT / FILE_LABS, "a") as f:
    f.write("\nuna riga aggiunta per sbaglio\n")
(REPO_ROOT / FILE_WORK).write_text("un file mio\n")
git("status", "--short")
git("diff", "--stat", FILE_LABS)
git("restore", FILE_LABS)
git("status", "--short")

**Che cosa guardare.** Il file di `labs/` compare con una `M` (modificato) e `git diff` mostra la riga aggiunta; il file in `work/` non compare affatto, perché `work/` è nel `.gitignore`. Dopo `git restore` il file di `labs/` torna com'era e `git status` è di nuovo vuoto. Per questo si lavora sempre in `work/`: un file modificato in `labs/` farebbe fallire il `git pull` della lezione successiva.

**Da discutere**

1. Che cosa succede alle modifiche di un file dopo `git restore`? E dopo `git stash`?
2. Perché `git pull` si rifiuta di aggiornare un file che hai modificato?

## Chiusura

**Il numero di oggi**: il ritardo mediano, in millisecondi, fra la pubblicazione di un
messaggio e la sua consegna da parte del broker sul proprio portatile, su 20 messaggi a QoS 0.

In [ ]:
ritardi = []
for i in range(20):
    m = un_messaggio("test/ritardo", f"messaggio {i}")
    if m:
        ritardi.append(m["ritardo_ms"])
print(f"{len(ritardi)} messaggi consegnati su 20; ritardo mediano {np.median(ritardi):.2f} ms")

Su `localhost` il messaggio non esce dal portatile: il ritardo misura solo il lavoro di client e
broker, ed è dell'ordine del millisecondo. Nel Lab 1 lo stesso scambio si misura byte per byte, e
nel Blocco 0.1 si vede quanto pesano le intestazioni su un collegamento lento.

**Da riportare**: il numero di oggi e l'uscita di `python scripts/verify_setup.py --quick`
(il file `verifica.txt`).